In [3]:
%load_ext autoreload
%autoreload 2

import sys, logging, warnings, os
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore", message=".*longdouble.*", category=UserWarning)
logging.basicConfig(level=logging.WARNING, format="%(levelname)-8s %(message)s")
#logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(message)s")
import pandas as pd
import numpy as np

from src import CVDPreventClient, CVDPApiError, TARGET_PERIOD_ID, CKD_INDICATOR_CODES, clean_raw_records


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Extract — fetch raw records from API

In [4]:
# Filter parameters — change these to explore other stratifications
# "Persons" / "Sex"  → ICB-level un-stratified total (used for this analysis)
# Other options the API returns: Age, Deprivation quintile, age-standardised variants
CATEGORY_ATTRIBUTE   = "Persons"
METRIC_CATEGORY_TYPE = "Sex"

client = CVDPreventClient()

records = client.fetch_ckd_icb_data(
    time_period_id=31,
    indicator_codes=["CVDP001CKD", "CVDP002CKD", "CVDP004CKD", "CVDP006CKD"],
)

filtered_records = [
    r for r in records
    if r["category_attribute"]         == CATEGORY_ATTRIBUTE
    and r["metric_category_type_name"] == METRIC_CATEGORY_TYPE
]

print(f"Total records (all categories): {len(records)}")
print(f"After filter ({CATEGORY_ATTRIBUTE!r} / {METRIC_CATEGORY_TYPE!r}): {len(filtered_records)}")


Total records (all categories): 4620
After filter ('Persons' / 'Sex'): 168


## 2. Clean — validate and normalise raw records

In [7]:
df_long = clean_raw_records(filtered_records)

# Long-format: one row per (ICB × indicator)

#df_long.head(3)

print(f"number of df_long records: {len(df_long)}")

number of df_long records: 168


## 3. Transform 

Each indicator has a `numerator` and `denominator`. We pivot so every ICB
becomes one row with one column per (indicator × num/den).

Note that business case state Denominator for CVDP002CKD is the total patients with eGFR test results that makes them eligible to be diagnosed with CKD , but it seems like it is same as CVDP001CKD which is total population, in order to find N patients eligible for diagnosis of CKD , I took numerator of CVDP001CKD (dianosed ckd) + numerator of CVDP002CKD (eligible but uncoded)


In [4]:
#Pivot: long → wide
#Index uniquely identifies one (ICB, period) row
index_cols = ["area_id","area_name","time_period_id","time_period_name"]

num_wide = df_long.pivot_table(index=index_cols, columns="indicator_code",
                               values="numerator", aggfunc="first")
num_wide.columns = [f"{c}_num" for c in num_wide.columns]

den_wide = df_long.pivot_table(index=index_cols, columns="indicator_code",
                               values="denominator", aggfunc="first")
den_wide.columns = [f"{c}_den" for c in den_wide.columns]

wide = pd.concat([num_wide, den_wide], axis=1).reset_index()

# Guard: fail clearly if any expected indicator column is missing
REQUIRED_COLS = ["CVDP001CKD_num", "CVDP001CKD_den",
                 "CVDP002CKD_num",
                 "CVDP004CKD_num", "CVDP004CKD_den",
                 "CVDP006CKD_num", "CVDP006CKD_den"]
missing = [c for c in REQUIRED_COLS if c not in wide.columns]
if missing:
    raise ValueError(f"Missing columns after pivot: {missing}")

wide = wide.dropna(subset=REQUIRED_COLS)


In [5]:
#Build business columns
df = pd.DataFrame()
df["ICB"]          = wide["area_name"]
df["Time Period"]  = wide["time_period_name"]

df["N total Population"]                    = wide["CVDP001CKD_den"].astype(int)
df["N patients eligible for CKD diagnosis"] = (wide["CVDP001CKD_num"] + wide["CVDP002CKD_num"]).astype(int)  # diagnosed + undiagnosed
df["N patients eligible but uncoded"]       = wide["CVDP002CKD_num"].astype(int)   # undiagnosed gap
df["N patients eligible for eGFR tests"]    = wide["CVDP006CKD_den"].astype(int)
df["N patients tested for eGFR"]            = wide["CVDP006CKD_num"].astype(int)
df["N patients eligible for uACR tests"]    = wide["CVDP004CKD_den"].astype(int)
df["N patients tested for uACR"]            = wide["CVDP004CKD_num"].astype(int)

df = df.sort_values("N total Population", ascending=False).reset_index(drop=True)
df.head(3)


,ICB,Time Period,N total Population,N patients eligible for CKD diagnosis,N patients eligible but uncoded,N patients eligible for eGFR tests,N patients tested for eGFR,N patients eligible for uACR tests,N patients tested for uACR
0,NHS North East and North Cumbria Integrated Ca...,To December 2025,2635055,152670,12880,139790,130420,139790,86720
1,NHS Greater Manchester Integrated Care Board,To December 2025,2594325,112500,6815,105685,95880,105685,55965
2,NHS North West London Integrated Care Board,To December 2025,2444745,63650,2370,61280,56425,61280,41130


In [6]:
#Sanity checks
checks = {
    "uncoded ≤ eligible for CKD":
        (df["N patients eligible but uncoded"] <= df["N patients eligible for CKD diagnosis"]).all(),
    "eGFR tested ≤ eGFR eligible":
        (df["N patients tested for eGFR"] <= df["N patients eligible for eGFR tests"]).all(),
    "uACR tested ≤ uACR eligible":
        (df["N patients tested for uACR"] <= df["N patients eligible for uACR tests"]).all(),
    "CKD eligible ≤ total population":
        (df["N patients eligible for CKD diagnosis"] <= df["N total Population"]).all(),
}
for desc, ok in checks.items():
    print(f"{'✓' if ok else '✗'} {desc}")


✓ uncoded ≤ eligible for CKD
✓ eGFR tested ≤ eGFR eligible
✓ uACR tested ≤ uACR eligible
✓ CKD eligible ≤ total population


---
## 4. Analysis

### Q1 — Which ICBs represent the largest market opportunity?

Ranking by absolute size of uncoded patients. Note that some ICBs have a higher uncoded percentage, so they may be more cost-effective to target even with a smaller absolute gap.


In [7]:
df["pct_uncoded"] = (
    df["N patients eligible but uncoded"] / df["N patients eligible for CKD diagnosis"] * 100
).round(3)

top10_abs = df.nlargest(10, "N patients eligible but uncoded")[
    ["ICB", "N total Population", "N patients eligible for CKD diagnosis", "N patients eligible but uncoded", "pct_uncoded"]
].reset_index(drop=True)

print("Top 10 ICBs — absolute undiagnosed gap")
top10_abs


Top 10 ICBs — absolute undiagnosed gap


,ICB,N total Population,N patients eligible for CKD diagnosis,N patients eligible but uncoded,pct_uncoded
0,NHS North East and North Cumbria Integrated Ca...,2635055,152670,12880,8.436
1,NHS Humber and North Yorkshire Integrated Care...,1485970,90985,11140,12.244
2,Nhs Hampshire And Isle Of Wight Integrated Car...,1604260,79465,9820,12.358
3,Nhs Sussex Integrated Care Board,1525190,87680,8085,9.221
4,NHS Hertfordshire and West Essex Integrated Ca...,1318720,56805,8075,14.215
5,NHS West Yorkshire Integrated Care Board,2132370,102610,7845,7.645
6,NHS Cheshire and Merseyside Integrated Care Board,2256830,116960,7760,6.635
7,"NHS Buckinghamshire, Oxfordshire and Berkshire...",1576605,65970,7705,11.680
8,NHS Kent and Medway Integrated Care Board,1606255,92420,7650,8.277
9,NHS Devon Integrated Care Board,1068350,65510,7380,11.265


### Q2 — Investment priority: market potential vs diagnostic gap

**Framework:**
The business case has an explicit constraint: *ICBs with low market potential are not worth investing in, even if their diagnostic rates are poor.*

This makes Q2 a **2-dimensional** decision:
- **Market potential (X)** = total ICB population — how large is the addressable market?
- **Diagnostic gap (Y)** = % uncoded = uncoded ÷ (diagnosed + uncoded) — how poor is the diagnosis rate?



In [8]:
df["egfr_rate"] = df["N patients tested for eGFR"] / df["N patients eligible for eGFR tests"]
df["uacr_rate"] = df["N patients tested for uACR"] / df["N patients eligible for uACR tests"]

# Quadrant thresholds: national medians (data-driven, no arbitrary cutoffs)
med_pop     = df["N total Population"].median()
med_pct_unc = df["pct_uncoded"].median()

hi_pop = df["N total Population"] >= med_pop
hi_gap = df["pct_uncoded"]        >= med_pct_unc

df["quadrant"] = np.select(
    condlist  = [hi_pop & hi_gap, hi_pop & ~hi_gap, ~hi_pop & hi_gap],
    choicelist= ["Priority (large market + high gap)", "Maintain (large market + low gap)", "Avoid (small market + high gap)"],
    default   = "Low priority",
)


#Priority table
print("\nPriority ICBs (large market + high diagnostic gap):")
df[df["quadrant"] == "Priority (large market + high gap)"][
    ["ICB", "N total Population", "N patients eligible but uncoded", "pct_uncoded"]
].sort_values("N patients eligible but uncoded", ascending=False).reset_index(drop=True)



Priority ICBs (large market + high diagnostic gap):


,ICB,N total Population,N patients eligible but uncoded,pct_uncoded
0,NHS North East and North Cumbria Integrated Ca...,2635055,12880,8.436
1,NHS Humber and North Yorkshire Integrated Care...,1485970,11140,12.244
2,Nhs Hampshire And Isle Of Wight Integrated Car...,1604260,9820,12.358
3,Nhs Sussex Integrated Care Board,1525190,8085,9.221
4,NHS Hertfordshire and West Essex Integrated Ca...,1318720,8075,14.215
5,"NHS Buckinghamshire, Oxfordshire and Berkshire...",1576605,7705,11.680
6,NHS Kent and Medway Integrated Care Board,1606255,7650,8.277
7,NHS Devon Integrated Care Board,1068350,7380,11.265
8,NHS North East London Integrated Care Board,1949645,7210,9.558
9,NHS Mid and South Essex Integrated Care Board,1021030,6890,12.067


### Q3 — Testing strategy: eGFR-led case-finding in priority ICBs

**Why eGFR, not uACR:**

Base on title in data explorer
CVDP002CKD: Patients whose last two eGFRs are less than 60ml/min/1.73m2 (uncoded CKD), who do not have a record of GP recorded CKD (G3a to G5)
CVDP004CKD: Patients with GP recorded CKD (G3a to G5) with a record of a urine ACR test in the preceding 12 months

so ACR test is only done for confirmed patient to check how bad is the CKD. In that sense if we are looking for expanding the market, existing patient can be ignored

**two approach, number of undiagnosed patient and test rate**
Uncoded patients already have eGFR evidence. The bottleneck is GPs not translating that evidence into a CKD code. The intervention is a **systematic eGFR recall and coding workflow**: proactively identify patients with historical eGFR < 60, invite them for a confirmatory test, and code those who meet the criteria.

**ICBs with below-median eGFR rates on their diagnosed population** are another priorities: their eGFR infrastructure is weaker across the board, meaning at-risk undiagnosed patients are also less likely to be captured. These ICBs need the most intensive eGFR-focused investment.

In [9]:
# Q3: eGFR testing culture in priority ICBs (top 10 by absolute uncoded count)
median_egfr = df["egfr_rate"].median()
print(f"National median eGFR rate (diagnosed CKD patients): {median_egfr:.1%}\n")

priority_icbs = df.nlargest(10, "N patients eligible but uncoded")["ICB"].tolist()

q3 = df[df["ICB"].isin(priority_icbs)].copy()
q3["eGFR rate %"]        = (q3["egfr_rate"] * 100).round(1)
q3["eGFR gap vs median"] = ((median_egfr - q3["egfr_rate"]) * 100).round(1)  # positive = below median
q3["Below median"]       = q3["eGFR gap vs median"] > 0

q3 = q3[[
    "ICB",
    "N patients eligible but uncoded",
    "pct_uncoded",
    "eGFR rate %",
    "eGFR gap vs median",
    "Below median",
]].rename(columns={
    "N patients eligible but uncoded": "N uncoded",
    "pct_uncoded": "% uncoded",
})

q3_sorted = q3.sort_values(["eGFR gap vs median", "N uncoded"], ascending=[False, False]).reset_index(drop=True)
print("eGFR testing culture in top-10 priority ICBs:")
q3_sorted


National median eGFR rate (diagnosed CKD patients): 91.1%

eGFR testing culture in top-10 priority ICBs:


,ICB,N uncoded,% uncoded,eGFR rate %,eGFR gap vs median,Below median
0,Nhs Hampshire And Isle Of Wight Integrated Car...,9820,12.358,82.3,8.9,True
1,NHS Kent and Medway Integrated Care Board,7650,8.277,88.6,2.5,True
2,"NHS Buckinghamshire, Oxfordshire and Berkshire...",7705,11.680,89.6,1.5,True
3,NHS Cheshire and Merseyside Integrated Care Board,7760,6.635,90.5,0.7,True
4,Nhs Sussex Integrated Care Board,8085,9.221,90.8,0.3,True
5,NHS Hertfordshire and West Essex Integrated Ca...,8075,14.215,91.1,0.0,False
6,NHS Humber and North Yorkshire Integrated Care...,11140,12.244,91.9,-0.8,False
7,NHS Devon Integrated Care Board,7380,11.265,92.2,-1.1,False
8,NHS West Yorkshire Integrated Care Board,7845,7.645,92.7,-1.5,False
9,NHS North East and North Cumbria Integrated Ca...,12880,8.436,93.3,-2.2,False


## 5. Bonus Analysis

### Bonus A — Dynamic Trends: are ICBs improving their diagnosis rates?


- **CVDP001CKD value** (% population diagnosed with CKD) — rising = more patients being coded
- **CVDP002CKD value** (% population eligible but uncoded) — falling = the gap is closing


In [10]:

# GET /indicator?timePeriodID=31&systemLevelID=7&areaID=X&tagID=9
# tagID=9 filters server-side to CKD indicators only (~6 vs ~60), reducing payload size
TS_CODES   = {"CVDP001CKD", "CVDP002CKD"}
CKD_TAG_ID = 9

# area_id → name map already in memory from the Clean step
area_lookup = (
    df_long[["area_id", "area_name"]].drop_duplicates()
    .set_index("area_id")["area_name"].to_dict()
)

ts_records = []
for area_id, area_name in area_lookup.items():
    #call api to get indictor with tag id = 9 (CKD only, fewer calls) and later filter indicator code
    for ind in client._get("/indicator", {
        "timePeriodID": TARGET_PERIOD_ID,
        "systemLevelID": 7,
        "areaID": area_id,
        "tagID":  CKD_TAG_ID,
    }).get("indicatorList", []):
        code = ind.get("IndicatorCode", "")
        if code in TS_CODES:
            for cat in ind.get("Categories", []):
                if cat.get("CategoryAttribute") == "Persons" and cat.get("MetricCategoryTypeName") == "Sex":
                    ts_records.extend(
                        {
                            "area_name":        area_name,
                            "indicator_code":   code,
                            "time_period_id":   ts["TimePeriodID"],
                            "time_period_name": ts["TimePeriodName"],
                            "value":            float(ts["Value"]),
                        }
                        for ts in cat.get("TimeSeries", []) if ts.get("Value") is not None
                    )

DISPLAY_COLS = ["ICB", "First period", "First value", "Last period", "Last value", "Change (pp)"]
#sort it for first/last in group by
ts_df = pd.DataFrame(ts_records).sort_values("time_period_id")
trend_summary = (
    ts_df
    .groupby(["indicator_code", "area_name"], sort=False)
    .agg(**{
        "First period": ("time_period_name", "first"),
        "First value":  ("value",            "first"),
        "Last period":  ("time_period_name", "last"),
        "Last value":   ("value",            "last"),
    })
    .assign(**{"Change (pp)": lambda d: (d["Last value"] - d["First value"]).round(2)})
    .round({"First value": 2, "Last value": 2})
    .reset_index()
    .rename(columns={"area_name": "ICB", "indicator_code": "Indicator"})
)

print("Coded CKD prevalence change (positive = improvement):")
display(trend_summary[trend_summary["Indicator"] == "CVDP001CKD"]
        .sort_values("Change (pp)", ascending=False)[DISPLAY_COLS].reset_index(drop=True))

print("\nUncoded gap change (negative = improvement = gap closing):")
display(trend_summary[trend_summary["Indicator"] == "CVDP002CKD"]
        .sort_values("Change (pp)")[DISPLAY_COLS].reset_index(drop=True))


Coded CKD prevalence change (positive = improvement):


,ICB,First period,First value,Last period,Last value,Change (pp)
0,"NHS Leicester, Leicestershire and Rutland Inte...",To June 2022,3.12,To December 2025,5.29,2.17
1,NHS Lincolnshire Integrated Care Board,To June 2022,6.10,To December 2025,7.91,1.81
2,NHS Suffolk and North East Essex Integrated Ca...,To June 2022,4.59,To December 2025,6.16,1.57
3,Nhs Norfolk And Waveney Integrated Care Board,To June 2022,4.01,To December 2025,5.56,1.55
4,NHS Mid and South Essex Integrated Care Board,To June 2022,3.46,To December 2025,4.92,1.46
5,NHS Cambridgeshire and Peterborough Integrated...,To June 2022,2.16,To December 2025,3.60,1.44
6,"NHS Bath and North East Somerset, Swindon and ...",To June 2022,3.27,To December 2025,4.63,1.36
7,NHS Dorset Integrated Care Board,To June 2022,4.87,To December 2025,6.22,1.35
8,NHS Northamptonshire Integrated Care Board,To June 2022,3.43,To December 2025,4.78,1.35
9,NHS West Yorkshire Integrated Care Board,To June 2022,3.22,To December 2025,4.44,1.22



Uncoded gap change (negative = improvement = gap closing):


,ICB,First period,First value,Last period,Last value,Change (pp)
0,NHS Coventry and Warwickshire Integrated Care ...,To June 2022,0.95,To December 2025,0.32,-0.63
1,NHS Suffolk and North East Essex Integrated Ca...,To June 2022,1.24,To December 2025,0.65,-0.59
2,Nhs Norfolk And Waveney Integrated Care Board,To June 2022,1.02,To December 2025,0.46,-0.56
3,NHS Lincolnshire Integrated Care Board,To June 2022,0.84,To December 2025,0.34,-0.50
4,NHS Derby and Derbyshire Integrated Care Board,To June 2022,0.88,To December 2025,0.39,-0.49
5,NHS Black Country Integrated Care Board,To June 2022,0.74,To December 2025,0.26,-0.48
6,NHS Cambridgeshire and Peterborough Integrated...,To June 2022,0.80,To December 2025,0.35,-0.45
7,NHS Mid and South Essex Integrated Care Board,To June 2022,1.11,To December 2025,0.68,-0.43
8,NHS Herefordshire and Worcestershire Integrate...,To June 2022,0.66,To December 2025,0.24,-0.42
9,NHS West Yorkshire Integrated Care Board,To June 2022,0.78,To December 2025,0.37,-0.41


### Bonus B — ICB Segmentation: unsupervised K-Means clustering

K-Means groups the 42 ICBs into clusters based on their overall profile across five features. This reveals whether ICBs that look similar on one dimension are also similar on others — and identifies distinct archetypes for targeted strategy.

**Features used (all StandardScaler-normalised):**
- `% uncoded` — size of the undiagnosed gap
- `eGFR rate` — diagnostic testing quality (proxy for testing culture)
- `uACR rate` — monitoring quality for existing patients
- `log(total population)` — market size (log-scaled to reduce skew)
- `log(N uncoded)` — absolute new-market opportunity

**k=3** selected via elbow (inertia) and silhouette score — sufficient to tell a clear strategic story without over-segmenting 42 ICBs.


In [14]:

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

features = pd.DataFrame({
    "pct_uncoded":  df["pct_uncoded"],
    "egfr_rate":    df["egfr_rate"],
    "uacr_rate":    df["uacr_rate"],
    "log_pop":      np.log(df["N total Population"]),
    "log_uncoded":  np.log(df["N patients eligible but uncoded"]),
})

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(features)

#k selection: silhouette score table (k=3 is the elbow + silhouette peak)
ks          = range(2, 8)
inertias    = []
silhouettes = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

k_selection = pd.DataFrame({"k": list(ks), "Inertia": [round(v, 1) for v in inertias],
                             "Silhouette score": [round(v, 3) for v in silhouettes]})
print("k selection — choose k where silhouette peaks:")
display(k_selection)

# Fit k=3
K = 3
km3           = KMeans(n_clusters=K, random_state=42, n_init=20)
df["cluster"] = km3.fit_predict(X_scaled)

# Centroid table (in original scale)
centroids_orig = scaler.inverse_transform(km3.cluster_centers_)
centroid_df = pd.DataFrame(centroids_orig, columns=features.columns)
centroid_df.index.name = "Cluster"
print("\nCluster centroids (original scale):")
display(centroid_df.round(3))

# Auto-label based on pct_uncoded centroid rank
order      = centroid_df["pct_uncoded"].rank().astype(int)
labels_map = {order.idxmin(): "Low gap", order.idxmax(): "High gap",
              [i for i in range(K) if i not in [order.idxmin(), order.idxmax()]][0]: "Mid gap"}
df["cluster_label"] = df["cluster"].map(labels_map)

CLUSTER_ORDER = ["High gap", "Mid gap", "Low gap"]

#Cluster summary
cluster_summary = df.groupby("cluster_label").agg(
    N_ICBs            = ("ICB",                             "count"),
    avg_pct_uncoded   = ("pct_uncoded",                     "mean"),
    avg_egfr_rate     = ("egfr_rate",                       "mean"),
    avg_uacr_rate     = ("uacr_rate",                       "mean"),
    median_population = ("N total Population",              "median"),
    total_uncoded     = ("N patients eligible but uncoded", "sum"),
).round(3)
cluster_summary["avg_egfr_rate"] = (cluster_summary["avg_egfr_rate"] * 100).round(1)
cluster_summary["avg_uacr_rate"] = (cluster_summary["avg_uacr_rate"] * 100).round(1)
cluster_summary = cluster_summary.reindex(CLUSTER_ORDER)
print("\nCluster profiles:")
display(cluster_summary)

#ICB membership table — sorted High gap → Mid gap → Low gap, then by % uncoded within each
membership = (
    df[["ICB", "cluster_label", "pct_uncoded", "N patients eligible but uncoded", "N total Population","egfr_rate", "uacr_rate"]]
    .rename(columns={"cluster_label": "Cluster", "pct_uncoded": "% uncoded",
                     "N patients eligible but uncoded": "N uncoded"})
    .copy()
)
membership["egfr_rate"] = membership["egfr_rate"].round(3)
membership["uacr_rate"] = membership["uacr_rate"].round(3)
membership["Cluster"] = pd.Categorical(membership["Cluster"], categories=CLUSTER_ORDER, ordered=True)
print("\nICB cluster membership:")
display(
    membership
    .sort_values(["Cluster", "% uncoded"], ascending=[True, False])
    .reset_index(drop=True)
)


k selection — choose k where silhouette peaks:


,k,Inertia,Silhouette score
0,2,142.8,0.279
1,3,112.9,0.259
2,4,96.5,0.224
3,5,83.4,0.237
4,6,73.0,0.209
5,7,61.6,0.246



Cluster centroids (original scale):


,pct_uncoded,egfr_rate,uacr_rate,log_pop,log_uncoded
Cluster,,,,,
0,10.320,0.883,0.420,13.982,8.695
1,6.650,0.920,0.538,13.612,7.924
2,9.529,0.913,0.563,14.252,8.845



Cluster profiles:


,N_ICBs,avg_pct_uncoded,avg_egfr_rate,avg_uacr_rate,median_population,total_uncoded
cluster_label,,,,,,
High gap,9,10.320,88.3,42.0,1276225.0,57390
Mid gap,14,9.529,91.3,56.3,1503805.0,101330
Low gap,19,6.650,92.0,53.8,869535.0,53800



ICB cluster membership:


,ICB,Cluster,% uncoded,N uncoded,N total Population,egfr_rate,uacr_rate
0,"NHS Bedfordshire, Luton and Milton Keynes Inte...",High gap,16.245,5970,892525,0.896,0.448
1,Nhs Hampshire And Isle Of Wight Integrated Car...,High gap,12.358,9820,1604260,0.823,0.448
2,NHS Mid and South Essex Integrated Care Board,High gap,12.067,6890,1021030,0.886,0.372
3,"NHS Buckinghamshire, Oxfordshire and Berkshire...",High gap,11.680,7705,1576605,0.896,0.446
4,NHS Dorset Integrated Care Board,High gap,9.711,4620,690675,0.910,0.398
5,Nhs Sussex Integrated Care Board,High gap,9.221,8085,1525190,0.908,0.442
6,NHS Kent and Medway Integrated Care Board,High gap,8.277,7650,1606255,0.886,0.422
7,NHS Birmingham and Solihull Integrated Care Board,High gap,7.381,3720,1276225,0.887,0.471
8,NHS Coventry and Warwickshire Integrated Care ...,High gap,5.941,2930,902450,0.853,0.337
9,NHS Hertfordshire and West Essex Integrated Ca...,Mid gap,14.215,8075,1318720,0.911,0.561


### Bonus C — The Role of Gender and Age

In [15]:
#C1: Gender analysis
sex_records = [
    r for r in records
    if r.get("metric_category_type_name") == "Sex"
    and r.get("category_attribute") in ("Male", "Female")
    and r.get("indicator_code") in ("CVDP001CKD", "CVDP002CKD")
]
sex_df = pd.DataFrame(sex_records)
sex_df["value"] = pd.to_numeric(sex_df["value"], errors="coerce")
sex_df = sex_df.dropna(subset=["value"])

# National median prevalence and uncoded gap by sex
sex_summary = sex_df.groupby(["indicator_code", "category_attribute"])["value"].median().unstack()
print("National median values by sex (%):")
print(sex_summary.round(3))

#C2: Age analysis — MetricCategoryName holds the age band
age_records = [
    r for r in records
    if r.get("metric_category_type_name") == "Age group"
    and r.get("category_attribute") == "Persons"   # un-stratified total per age band
    and r.get("indicator_code") == "CVDP001CKD"
    and r.get("metric_category_name") is not None
]
age_df = pd.DataFrame(age_records)
age_df["value"] = pd.to_numeric(age_df["value"], errors="coerce")
age_df = age_df.dropna(subset=["value"])

age_summary = (
    age_df.groupby("metric_category_name")["value"]
    .median()
    .reset_index()
    .rename(columns={"metric_category_name": "Age band", "value": "Median CKD prevalence %"})
)
age_summary["sort_key"] = age_summary["Age band"].str.extract(r"(\d+)").astype(float)
age_summary = age_summary.sort_values("sort_key")


print("\nCKD prevalence by age band (national median across ICBs):")
print(age_summary[["Age band","Median CKD prevalence %"]].to_string(index=False))


National median values by sex (%):
category_attribute  Female   Male
indicator_code                   
CVDP001CKD           5.440  4.235
CVDP002CKD           0.435  0.360

CKD prevalence by age band (national median across ICBs):
Age band  Median CKD prevalence %
   18-39                    0.090
   40-59                    0.900
   60-79                    8.675
     80+                   33.630


In [ ]:
# C3: Aging population — which ICBs have the oldest demographics?
# Total-population lookup: denominator of CVDP001CKD 
pop_lookup = (
    df_long[df_long["indicator_code"] == "CVDP001CKD"]
    [["area_id", "area_name", "denominator"]]
    .rename(columns={"denominator": "icb_total_population"})
)

age_df_with_pop = age_df.merge(pop_lookup, on=["area_id", "area_name"], how="left")

age_summary_long = (
    age_df_with_pop
    .groupby(["metric_category_name", "area_name"])
    .apply(lambda g: pd.Series({
        "N in age band":    int(g["denominator"].iloc[0]),
        "N total ICB pop":  int(g["icb_total_population"].iloc[0]),
    }), include_groups=False)
    .reset_index()
    .rename(columns={"metric_category_name": "Age band"})
)

age_pivot = (
    age_summary_long
    .pivot(index="area_name", columns="Age band", values="N in age band")
    .rename_axis(columns=None)   # ← clears the "Age band" columns-axis label
    .reset_index()
    .rename(columns={"area_name": "ICB"})
)

age_cols = sorted(
    [c for c in age_pivot.columns if c != "ICB"],
    key=lambda x: float(str(x).split("-")[0].replace("+", ""))
)
age_pivot = age_pivot[["ICB"] + age_cols]

sixty_plus = [c for c in age_cols if float(str(c).split("-")[0].replace("+", "")) >= 60]
#sum horizontal across the 60+ age bands to get a single "60+ combined" column for sorting and display
age_pivot["60+ combined"] = age_pivot[sixty_plus].sum(axis=1)

display_tbl = (
    age_pivot
    .sort_values("60+ combined", ascending=False)
    .reset_index(drop=True)
    .copy()
)

print("ICB age-band distribution (absolute population counts) — sorted by 60+ count:")
display(display_tbl)


ICB age-band distribution (absolute population counts) — sorted by 60+ count:


,ICB,18-39,40-59,60-79,80+,60+ combined
0,NHS North East and North Cumbria Integrated Ca...,926995,803870,728395,175795,904190
1,NHS Cheshire and Merseyside Integrated Care Board,805020,707490,595510,148810,744320
2,NHS Greater Manchester Integrated Care Board,1085705,829795,549995,128830,678825
3,NHS West Yorkshire Integrated Care Board,843375,685815,485085,118095,603180
4,Nhs Hampshire And Isle Of Wight Integrated Car...,554570,499545,427270,122880,550150
5,NHS Humber and North Yorkshire Integrated Care...,478690,458340,434855,114085,548940
6,Nhs Sussex Integrated Care Board,495055,485965,419850,124320,544170
7,NHS Kent and Medway Integrated Care Board,548200,528685,416520,112850,529370
8,NHS Lancashire and South Cumbria Integrated Ca...,515765,465050,405170,103110,508280
9,NHS North West London Integrated Care Board,1180380,793345,387395,83620,471015


In [17]:
# C4: Age × Sex breakdown — absolute population counts per (age band × sex) across all ICBs

sex_age_records = [
    r for r in records
    if r.get("metric_category_type_name") == "Age group"
    and r.get("category_attribute") in ("Male", "Female")
    and r.get("indicator_code") == "CVDP001CKD"
]

sex_age_df = pd.DataFrame(sex_age_records)
sex_age_df["denominator"] = pd.to_numeric(sex_age_df["denominator"], errors="coerce")
sex_age_df = sex_age_df.dropna(subset=["denominator"])
sex_age_df["denominator"] = sex_age_df["denominator"].astype(int)

# One row per (ICB × age band × sex)
sex_age_pop = (
    sex_age_df
    .groupby(["area_name", "metric_category_name", "category_attribute"])["denominator"]
    .first()
    .reset_index()
    .rename(columns={
        "metric_category_name": "Age band",
        "category_attribute":   "Sex",
        "denominator":          "N",
    })
)

# Age bands in chronological order
age_bands = sorted(
    sex_age_pop["Age band"].unique(),
    key=lambda x: float(str(x).split("-")[0].replace("+", ""))
)
sixty_plus_bands = [b for b in age_bands if float(str(b).split("-")[0].replace("+", "")) >= 60]

# Build wide table directly — no pivot_table, no MultiIndex
rows = []
for icb, grp in sex_age_pop.groupby("area_name"):
    row = {"ICB": icb}
    for sex in ["Female", "Male"]:
        sex_data = grp[grp["Sex"] == sex].set_index("Age band")["N"]
        for band in age_bands:
            row[f"{sex} {band}"] = sex_data.get(band, 0)
        row[f"{sex} 60+ combined"] = sum(sex_data.get(b, 0) for b in sixty_plus_bands)
    rows.append(row)

display_tbl = (
    pd.DataFrame(rows)
    .merge(df[["ICB", "N total Population"]], on="ICB", how="left")
    .assign(**{"60+ combined (F+M)": lambda t: t["Female 60+ combined"] + t["Male 60+ combined"]})
    .sort_values("60+ combined (F+M)", ascending=False)
    .reset_index(drop=True)
)

print("Age × Sex population breakdown (absolute counts) — sorted by 60+ count:")
display(display_tbl)


Age × Sex population breakdown (absolute counts) — sorted by 60+ count:


,ICB,Female 18-39,Female 40-59,Female 60-79,Female 80+,Female 60+ combined,Male 18-39,Male 40-59,Male 60-79,Male 80+,Male 60+ combined,N total Population,60+ combined (F+M)
0,NHS North East and North Cumbria Integrated Ca...,450305,397030,373450,102410,475860,476695,406845,354950,73385,428335,2635055,904195
1,NHS Cheshire and Merseyside Integrated Care Board,395235,347735,305885,86770,392655,409785,359750,289625,62040,351665,2256830,744320
2,NHS Greater Manchester Integrated Care Board,534840,396675,278810,75695,354505,550865,433120,271185,53130,324315,2594325,678820
3,NHS West Yorkshire Integrated Care Board,414100,331810,248210,69190,317400,429270,354005,236880,48905,285785,2132370,603185
4,Nhs Hampshire And Isle Of Wight Integrated Car...,273000,246980,220445,71295,291740,281570,252565,206820,51585,258405,1604260,550145
5,NHS Humber and North Yorkshire Integrated Care...,237185,227425,222475,65915,288390,241510,230915,212380,48170,260550,1485970,548940
6,Nhs Sussex Integrated Care Board,245475,241335,218095,72995,291090,249580,244630,201755,51325,253080,1525190,544170
7,NHS Kent and Medway Integrated Care Board,273895,264965,215940,65680,281620,274300,263720,200580,47170,247750,1606255,529370
8,NHS Lancashire and South Cumbria Integrated Ca...,251885,228380,205835,59430,265265,263880,236670,199335,43680,243015,1489095,508280
9,NHS North West London Integrated Care Board,567085,358095,196540,48765,245305,613295,435250,190855,34855,225710,2444745,471015
